# 04 – Classification
**Practicals 12–15:** Logistic Regression, KNN, Naïve Bayes and SVM for predicting **high-waste meals**, compared with cross-validation.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets us import from the src/ folder
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.preprocessing import load_clean_data
from src.classification import (get_clf_data, get_models, compare_classifiers, cv_predictions)

df = load_clean_data()

## Attempt 1 – demand-side inputs only
Day, meal, menu, exam week, event, rain, temperature and registered students. `leftover_kg` is never used because the label is built from it (data leakage).

In [ ]:
X1, y1, num1 = get_clf_data(df, include_plan=False)
print("High-waste share:", round(y1.mean(), 2))
attempt1 = compare_classifiers(X1, y1, num1)
attempt1

## Attempt 2 – add the planned quantity
The kitchen decides how much to cook *before* the meal, so `plates_cooked` and `cook_ratio` (plates per registered student) are legitimate inputs for the question: *given today's plan, will waste be high?*

In [ ]:
X2, y2, num2 = get_clf_data(df, include_plan=True)
attempt2 = compare_classifiers(X2, y2, num2)
attempt2

**How to read the table:** the *always No* baseline shows what a useless model scores. Accuracy alone can mislead, so recall and F1 matter more here: missing a high-waste meal costs more than a false alarm.

## Confusion matrix of the best model

In [ ]:
best = attempt2.sort_values("F1", ascending=False).iloc[0]["Model"]
print("Best model by F1:", best)

pred = cv_predictions(X2, y2, num2, get_models()[best])
ConfusionMatrixDisplay(confusion_matrix(y2, pred),
                       display_labels=["Low waste", "High waste"]).plot()
plt.title(f"{best} confusion matrix (cross-validated)")
plt.savefig(FIG_DIR / "classification_confusion_matrix.png", dpi=120)
plt.show()

### My notes
*(Write what you found: which model won, whether any model beat the baseline on accuracy, and why recall matters for Annapurna.)*